# 02 — Baseline Evaluation

Run Qwen2.5-Coder 0.5B and 1.5B (base, no fine-tuning) on BIRD dev.
Modes: greedy (E0) and SC@16 (E1). Compare with published numbers.

In [ ]:
import os, sys, json, time
from pathlib import Path

REPO_DIR = Path.cwd()
if 'SLM-PRJ' not in str(REPO_DIR):
    REPO_DIR = Path('/kaggle/working/SLM-PRJ')
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

DATA_DIR = REPO_DIR.parent / 'data'
BIRD_DIR = DATA_DIR / 'bird'
RESULTS_DIR = REPO_DIR / 'results' / 'baselines'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from benchmarks.bird_loader import load_bird_dev, get_db_path, get_schema

bird_dev = load_bird_dev(BIRD_DIR)
print(f'BIRD dev: {len(bird_dev)} questions')

## 1. Load model with vLLM

In [ ]:
from vllm import LLM, SamplingParams

MODEL_NAME = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'  # or 0.5B

llm = LLM(
    model=MODEL_NAME,
    dtype='float16',
    max_model_len=2048,
    gpu_memory_utilization=0.85,
)
tokenizer = llm.get_tokenizer()
print(f'Loaded {MODEL_NAME}')

In [ ]:
from training.prompt_template import SYSTEM_MESSAGE
from core.prompt_builder import build_prompt

def make_chat_prompt(question, schema, evidence=''):
    messages = [
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user', 'content': build_prompt(question, schema, evidence)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

prompts = []
for r in bird_dev:
    schema = get_schema(BIRD_DIR, r['db_id'])
    prompts.append(make_chat_prompt(r['question'], schema, r.get('evidence', '')))

print(f'Built {len(prompts)} prompts')
print(f'Example prompt length: {len(prompts[0])} chars')

## 2. E0 — Greedy decoding

In [ ]:
from core.sql_extractor import extract_sql
from benchmarks.eval_ex import execution_match

greedy_params = SamplingParams(temperature=0.0, max_tokens=512)

start = time.time()
outputs = llm.generate(prompts, greedy_params)
elapsed = time.time() - start
print(f'Greedy generation: {elapsed:.1f}s ({elapsed/len(prompts):.2f}s/query)')

greedy_results = []
correct = 0
for i, (r, out) in enumerate(zip(bird_dev, outputs)):
    raw = out.outputs[0].text
    try:
        pred_sql = extract_sql(raw)
    except ValueError:
        pred_sql = ''
    db = get_db_path(BIRD_DIR, r['db_id'])
    matched = execution_match(pred_sql, r['gold_sql'], db) if pred_sql else False
    correct += int(matched)
    greedy_results.append({
        'question': r['question'], 'db_id': r['db_id'],
        'difficulty': r['difficulty'],
        'pred_sql': pred_sql, 'gold_sql': r['gold_sql'],
        'correct': matched,
    })

ex = correct / len(bird_dev)
print(f'E0 Greedy EX: {ex:.4f} ({correct}/{len(bird_dev)})')
print(f'Published 1.5B base greedy: ~28.40%')

with open(RESULTS_DIR / 'E0_greedy.json', 'w') as f:
    json.dump({'accuracy': ex, 'total': len(bird_dev), 'correct': correct,
               'model': MODEL_NAME, 'elapsed_s': elapsed}, f, indent=2)

## 3. E1 — Self-Consistency @ 16

In [ ]:
from collections import Counter
from core.result_normalizer import fingerprint, requires_order
from core.schema_inspector import open_readonly
import sqlite3

N_SAMPLES = 16
sc_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=512, n=N_SAMPLES)

start = time.time()
sc_outputs = llm.generate(prompts, sc_params)
sc_elapsed = time.time() - start
print(f'SC@{N_SAMPLES} generation: {sc_elapsed:.1f}s')

def exec_fingerprint(sql, db_path, timeout=30):
    try:
        conn = open_readonly(db_path)
        cursor = conn.execute(sql)
        rows = cursor.fetchall()
        cols = len(cursor.description)
        conn.close()
        ordered = requires_order(sql)
        return fingerprint(rows, cols, ordered)
    except Exception:
        return None

sc_correct = 0
sc_results = []
for i, (r, out) in enumerate(zip(bird_dev, sc_outputs)):
    db = get_db_path(BIRD_DIR, r['db_id'])
    candidates = []
    for o in out.outputs:
        try:
            sql = extract_sql(o.text)
        except ValueError:
            continue
        fp = exec_fingerprint(sql, db)
        if fp is not None:
            candidates.append((sql, fp))

    if candidates:
        vote = Counter(fp for _, fp in candidates)
        best_fp = vote.most_common(1)[0][0]
        best_sql = next(sql for sql, fp in candidates if fp == best_fp)
        matched = execution_match(best_sql, r['gold_sql'], db)
    else:
        best_sql = ''
        matched = False

    sc_correct += int(matched)
    sc_results.append({
        'question': r['question'], 'db_id': r['db_id'],
        'difficulty': r['difficulty'],
        'pred_sql': best_sql, 'gold_sql': r['gold_sql'],
        'correct': matched, 'n_valid_candidates': len(candidates),
    })

sc_ex = sc_correct / len(bird_dev)
print(f'E1 SC@{N_SAMPLES} EX: {sc_ex:.4f} ({sc_correct}/{len(bird_dev)})')

with open(RESULTS_DIR / 'E1_sc16.json', 'w') as f:
    json.dump({'accuracy': sc_ex, 'total': len(bird_dev), 'correct': sc_correct,
               'model': MODEL_NAME, 'n_samples': N_SAMPLES, 'elapsed_s': sc_elapsed}, f, indent=2)

## 4. Pilot — GPU throughput measurement

In [ ]:
import torch

if torch.cuda.is_available():
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    print(f'Peak VRAM: {peak_mb:.0f} MB')

print(f'\nThroughput summary:')
print(f'  Greedy: {len(prompts)/elapsed:.1f} queries/s')
print(f'  SC@{N_SAMPLES}: {len(prompts)/sc_elapsed:.1f} queries/s (x{N_SAMPLES} samples each)')
print(f'  SC@{N_SAMPLES} total samples: {len(prompts)*N_SAMPLES/sc_elapsed:.1f} samples/s')

## 5. Check author checkpoint (optional)

In [ ]:
# Check if CycloneBoy/slm_sql checkpoint is available on HuggingFace
try:
    from huggingface_hub import list_repo_files
    files = list_repo_files('CycloneBoy/slm_sql')
    print(f'Author checkpoint found: {len(files)} files')
    print('Files:', files[:10])
except Exception as e:
    print(f'Author checkpoint not available: {e}')
    print('Using published numbers as ceiling reference.')